In [ ]:
# # install octave
# !sudo apt-get -qq update
# !sudo apt-get -qq install octave octave-signal liboctave-dev

# # install oct2py that compatible with colab
# import os

# from pkg_resources import get_distribution

# os.system(
#     f"pip install -qq"
#     f" ipykernel=={get_distribution('ipykernel').version}"
#     f" ipython=={get_distribution('ipython').version}"
#     f" tornado=={get_distribution('tornado').version}"
#     f" oct2py"
# )

# # install packages
# !pip install -qq matpower matpowercaseframes

In [ ]:
import oct2py

import matpower

print(f"oct2py version: {oct2py.__version__}")
print(f"matpower version: {matpower.__version__}")

oct2py version: 5.8.0
matpower version: 8.1.0.2.2.4


In [ ]:
import pandas as pd
from matpowercaseframes.constants import COLUMNS

from matpower import start_instance

In [ ]:
m = start_instance(engine="octave")

## `runopf` wrapper to avoid `TypeError`

In [ ]:
DEFAULT_MPC_FIELDS = ["baseMVA", "version", "bus", "gen", "branch", "gencost"]


def run_octave_cmd(cmd, m=None, fields=None, verbose=False, **kwargs):
    """
    Generic wrapper to run a MATPOWER command and extract struct fields using Octave
    backend.

    Parameters
    ----------
    cmd : str
        Full Octave command string, e.g. "runpf(mpc, mpopt)".
    m : oct2py instance, optional
        If None, a new instance is started and shut down after.
    fields : list of str, optional
        Struct field names to extract from result. Defaults to DEFAULT_MPC_FIELDS.
    verbose : bool, optional
        Verbosity of oct2py eval, by default False.
    **kwargs :
        Variables pushed into Octave session before running cmd.
        Keys become Octave variable names, e.g. mpc=mpc, mpopt=mpopt.
    """
    if fields is None:
        fields = DEFAULT_MPC_FIELDS

    if m is None:
        m = start_instance(engine="octave")
        SHUTDOWN = True
    else:
        SHUTDOWN = False

    for var_name, value in kwargs.items():
        m.push(var_name, value)
    m.eval(f"r1_ = {cmd};", verbose=verbose)

    result = {}
    for field in fields:
        result[field] = m.eval(f"r1_.{field};", verbose=verbose)

    if SHUTDOWN:
        m.exit()

    return result

In [ ]:
mpc = m.loadcase("case9")
display(pd.DataFrame(mpc["gen"], columns=COLUMNS["gen"][: mpc["gen"].shape[1]]))

,GEN_BUS,PG,QG,QMAX,QMIN,VG,MBASE,GEN_STATUS,PMAX,PMIN,...,PC2,QC1MIN,QC1MAX,QC2MIN,QC2MAX,RAMP_AGC,RAMP_10,RAMP_30,RAMP_Q,APF
0,1.0,72.3,27.03,300.0,-300.0,1.040,100.0,1.0,250.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2.0,163.0,6.54,300.0,-300.0,1.025,100.0,1.0,300.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3.0,85.0,-10.95,300.0,-300.0,1.025,100.0,1.0,270.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
mpc = run_octave_cmd("runopf(mpc)", m=m, mpc=mpc)

In [ ]:
display(pd.DataFrame(mpc["gen"], columns=COLUMNS["gen"]))

,GEN_BUS,PG,QG,QMAX,QMIN,VG,MBASE,GEN_STATUS,PMAX,PMIN,...,QC2MAX,RAMP_AGC,RAMP_10,RAMP_30,RAMP_Q,APF,MU_PMAX,MU_PMIN,MU_QMAX,MU_QMIN
0,1.0,89.798614,12.938736,300.0,-300.0,1.099951,100.0,1.0,250.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2.0,134.320652,0.047730,300.0,-300.0,1.097363,100.0,1.0,300.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3.0,94.187439,-22.619730,300.0,-300.0,1.086627,100.0,1.0,270.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
